# Directional Stock Prediction Using News, Sentiment, and Semantic Web Mining

**Group Members:** (Insert names)

---

## Abstract

This project develops an interpretable framework for predicting whether the next trading day's closing price for **Apple (AAPL)** or **Amazon (AMZN)** will move up or down. The system integrates TF-IDF lexical features, VADER sentiment scores, and semantic features from Named Entity Recognition (NER) and an RDF Knowledge Graph. A leakage-free timestamp rule ensures only news published before market close is used. Logistic Regression with L1/L2 regularization provides interpretability. Evaluation uses chronological splits, TimeSeriesSplit, and an ablation study.

**Keywords:** stock prediction, TF-IDF, sentiment analysis, semantic web, Logistic Regression, knowledge graph

---

## Problem Definition

Financial markets react to narratives embedded in news. This project builds an interpretable pipeline that:
- Aligns news to trading days **without look-ahead bias** (4 PM EST rule)
- Extracts **lexical**, **sentiment**, and **semantic** features
- Trains a **Logistic Regression** classifier
- Constructs a **Knowledge Graph** for explainability

**Binary classification:**

$$y_t = \begin{cases} 1 & \text{if } \text{Close}_{t+1} > \text{Close}_t \\ 0 & \text{otherwise} \end{cases}$$

## Section 1: Import Required Libraries

In [ ]:
# ── Core ──────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# ── Timezone handling ──────────────────────────────────────────────────────────
import pytz

# ── Stock data ─────────────────────────────────────────────────────────────────
import yfinance as yf

# ── NLP / Lexical ──────────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2

# ── Sentiment ──────────────────────────────────────────────────────────────────
import nltk
nltk.download("vader_lexicon", quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# ── Semantic / NER ─────────────────────────────────────────────────────────────
import spacy
# python -m spacy download en_core_web_sm  (run once)
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download as spacy_download
    spacy_download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

# ── Knowledge Graph ────────────────────────────────────────────────────────────
from rdflib import Graph, Literal, Namespace, RDF, URIRef
from rdflib.namespace import RDFS, OWL, XSD
from rdflib.plugins.sparql import prepareQuery

# ── Modeling & Evaluation ─────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from scipy.sparse import hstack, csr_matrix

# ── Visualization ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx

# ── Settings ───────────────────────────────────────────────────────────────────
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid", palette="muted")
RANDOM_STATE = 42
TICKERS = ["AAPL", "AMZN"]

print("✅ All libraries loaded successfully.")

## Section 2: Data Collection — News Data

Load the NASDAQ-style news CSV archive. Expected columns: `headline`, `content`, `published_at`, `source`.  
Update `NEWS_CSV_PATH` to point to your local file.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Update this path to your actual CSV file.
NEWS_CSV_PATH = "news_data.csv"   # <-- CHANGE ME

# ── Load ───────────────────────────────────────────────────────────────────────
try:
    news_raw = pd.read_csv(NEWS_CSV_PATH, parse_dates=["published_at"])
    print(f"✅ Loaded {len(news_raw):,} rows from {NEWS_CSV_PATH}")
except FileNotFoundError:
    # ── Synthetic fallback so the notebook runs end-to-end ────────────────────
    print("⚠️  News CSV not found — generating synthetic demo data.")
    rng = np.random.default_rng(42)
    n = 2000
    start = pd.Timestamp("2018-01-01", tz="UTC")
    end   = pd.Timestamp("2019-03-01", tz="UTC")
    timestamps = pd.to_datetime(
        rng.integers(start.value, end.value, size=n)
    ).tz_localize("UTC")
    tickers_col = rng.choice(TICKERS, size=n)
    words = ["earnings", "revenue", "growth", "loss", "acquisition",
             "product", "market", "investor", "guidance", "profit",
             "supply", "demand", "CEO", "quarterly", "beat", "miss"]
    headlines = [" ".join(rng.choice(words, size=7)) for _ in range(n)]
    contents  = [" ".join(rng.choice(words, size=30)) for _ in range(n)]
    news_raw = pd.DataFrame({
        "headline":     headlines,
        "content":      contents,
        "published_at": timestamps,
        "source":       rng.choice(["Reuters", "Bloomberg", "AP", "CNBC"], size=n),
        "ticker":       tickers_col,
    })

# ── Basic cleaning ─────────────────────────────────────────────────────────────
REQUIRED_COLS = {"headline", "content", "published_at"}
missing = REQUIRED_COLS - set(news_raw.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

news_raw["headline"] = news_raw["headline"].fillna("").str.strip()
news_raw["content"]  = news_raw["content"].fillna("").str.strip()
news_raw["text"]     = news_raw["headline"] + " " + news_raw["content"]

# ── Optional: filter to tickers of interest ───────────────────────────────────
if "ticker" in news_raw.columns:
    news_raw = news_raw[news_raw["ticker"].isin(TICKERS)].copy()
    print(f"   After ticker filter: {len(news_raw):,} rows")

print("\nSchema:")
print(news_raw.dtypes)
print("\nSample rows:")
news_raw.head(3)

## Section 3: Data Collection — Stock Price Data (Yahoo Finance)

Download daily OHLCV data for AAPL and AMZN from **Jan 2018 – Feb 2019** via `yfinance`.

In [ ]:
START_DATE = "2018-01-01"
END_DATE   = "2019-03-01"   # slight buffer beyond Feb 2019 to capture last label

stock_frames = {}
for ticker in TICKERS:
    df = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False, auto_adjust=True)
    df.index = pd.to_datetime(df.index)
    # Flatten multi-level columns if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]
    df = df[["Open", "High", "Low", "Close", "Volume"]].copy()
    df.index.name = "date"
    stock_frames[ticker] = df
    print(f"{ticker}: {df.shape[0]} trading days | {df.index.min().date()} → {df.index.max().date()}")
    print(df.head(3), "\n")

# Quick missing-day check
for ticker, df in stock_frames.items():
    nulls = df["Close"].isna().sum()
    print(f"{ticker} missing Close values: {nulls}")

## Section 4: Leakage-Free Timestamp Alignment (4 PM EST Rule)

**Rule:**  
- News published **before 4:00 PM EST** on trading day $t$ → assigned to day $t$  
- News published **at or after 4:00 PM EST** → assigned to the **next trading day**

This prevents any look-ahead bias from after-hours news influencing same-day labels.

In [ ]:
EST = pytz.timezone("America/New_York")
MARKET_CLOSE_HOUR = 16  # 4 PM

def assign_trading_day(ts: pd.Timestamp) -> pd.Timestamp:
    """
    Convert UTC timestamp to EST and apply the 4 PM rule.
    Returns the trading day (date only, tz-naive) to which this article belongs.
    """
    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")
    ts_est = ts.tz_convert(EST)
    if ts_est.hour < MARKET_CLOSE_HOUR:
        return ts_est.normalize().tz_localize(None)   # same calendar day
    else:
        return (ts_est + pd.Timedelta(days=1)).normalize().tz_localize(None)

# ── Normalise timestamps ───────────────────────────────────────────────────────
if news_raw["published_at"].dt.tz is None:
    news_raw["published_at"] = news_raw["published_at"].dt.tz_localize("UTC")

news_raw["trading_day"] = news_raw["published_at"].apply(assign_trading_day)

# ── Aggregate text per (trading_day, ticker) ───────────────────────────────────
ticker_col = "ticker" if "ticker" in news_raw.columns else None

if ticker_col:
    aligned = (
        news_raw.groupby([ticker_col, "trading_day"])["text"]
        .apply(lambda x: " ".join(x))
        .reset_index()
        .rename(columns={"text": "daily_text", ticker_col: "ticker"})
    )
else:
    # If no ticker column, duplicate for both tickers as a demo
    agg = (
        news_raw.groupby("trading_day")["text"]
        .apply(lambda x: " ".join(x))
        .reset_index()
        .rename(columns={"text": "daily_text"})
    )
    aligned = pd.concat([agg.assign(ticker=t) for t in TICKERS], ignore_index=True)

aligned["trading_day"] = pd.to_datetime(aligned["trading_day"])
print(f"Aligned shape: {aligned.shape}")
aligned.head()

## Section 5: Label Generation (Directional Binary Target)

$$y_t = \begin{cases} 1 & \text{if } \text{Close}_{t+1} > \text{Close}_t \quad (\text{price goes UP}) \\ 0 & \text{otherwise} \end{cases}$$

In [ ]:
labeled_frames = {}

for ticker in TICKERS:
    df = stock_frames[ticker].copy()
    df.index = pd.to_datetime(df.index).normalize()   # tz-naive date
    df["label"] = (df["Close"].shift(-1) > df["Close"]).astype(int)
    df = df.dropna(subset=["label"])
    df["ticker"] = ticker
    df = df.reset_index().rename(columns={"date": "trading_day"})

    # Merge with aligned news
    merged = pd.merge(
        df[["trading_day", "ticker", "Close", "label"]],
        aligned[aligned["ticker"] == ticker],
        on=["trading_day", "ticker"],
        how="inner"
    )
    labeled_frames[ticker] = merged.sort_values("trading_day").reset_index(drop=True)
    print(f"\n{ticker}: {len(merged)} days with news coverage")
    print(merged["label"].value_counts().rename({0: "DOWN (0)", 1: "UP (1)"}))

# ── Combined DataFrame for downstream processing ───────────────────────────────
master_df = pd.concat(labeled_frames.values(), ignore_index=True)
print(f"\nCombined master_df: {master_df.shape}")
master_df.head()

## Section 6: TF-IDF Lexical Feature Extraction

- Unigrams + bigrams (`ngram_range=(1,2)`)  
- English stopword removal  
- Sublinear TF scaling (`sublinear_tf=True`)

In [ ]:
tfidf_vec = TfidfVectorizer(
    ngram_range=(1, 2),
    stop_words="english",
    sublinear_tf=True,
    max_features=50_000,   # vocabulary cap before chi-sq reduction
    min_df=2,
)

corpus = master_df["daily_text"].tolist()
X_tfidf = tfidf_vec.fit_transform(corpus)

print(f"TF-IDF matrix shape  : {X_tfidf.shape}")
print(f"Vocabulary size      : {len(tfidf_vec.vocabulary_):,}")
print(f"Sample features      : {list(tfidf_vec.get_feature_names_out()[:20])}")

## Section 7: Chi-Square Feature Selection

Reduce the TF-IDF matrix to the **top 2,000 features** most associated with the direction label using $\chi^2$ selection.

In [ ]:
K_BEST = min(2000, X_tfidf.shape[1])   # guard if vocabulary < 2000
y_all  = master_df["label"].values

chi2_selector = SelectKBest(chi2, k=K_BEST)
X_tfidf_sel   = chi2_selector.fit_transform(X_tfidf, y_all)

# Retrieve selected feature names and scores
all_features   = tfidf_vec.get_feature_names_out()
support_mask   = chi2_selector.get_support()
selected_names = all_features[support_mask]
chi2_scores    = chi2_selector.scores_[support_mask]

top_df = (
    pd.DataFrame({"feature": selected_names, "chi2_score": chi2_scores})
    .sort_values("chi2_score", ascending=False)
    .reset_index(drop=True)
)

print(f"Selected TF-IDF features: {X_tfidf_sel.shape[1]}")
print("\nTop 20 chi-square features:")
top_df.head(20)

## Section 8: VADER Sentiment Feature Extraction

Compute **daily averages** of VADER scores (`compound`, `pos`, `neg`, `neu`) across all articles assigned to each trading day.

In [ ]:
sia = SentimentIntensityAnalyzer()

def vader_scores(text: str) -> dict:
    scores = sia.polarity_scores(str(text))
    return {
        "vader_compound": scores["compound"],
        "vader_pos":      scores["pos"],
        "vader_neg":      scores["neg"],
        "vader_neu":      scores["neu"],
    }

# Apply VADER to each article-level row, then aggregate per trading day + ticker
article_sentiment = news_raw[["trading_day", "ticker", "text"]].copy() if "ticker" in news_raw.columns else news_raw[["trading_day", "text"]].assign(ticker="ALL")

vader_rows = article_sentiment["text"].apply(vader_scores)
article_sentiment = pd.concat(
    [article_sentiment.reset_index(drop=True), pd.DataFrame(vader_rows.tolist())],
    axis=1
)

sent_daily = (
    article_sentiment
    .groupby(["ticker", "trading_day"])[["vader_compound", "vader_pos", "vader_neg", "vader_neu"]]
    .mean()
    .reset_index()
)

# ── Merge into master_df ───────────────────────────────────────────────────────
master_df = pd.merge(master_df, sent_daily, on=["ticker", "trading_day"], how="left")
master_df[["vader_compound","vader_pos","vader_neg","vader_neu"]] = (
    master_df[["vader_compound","vader_pos","vader_neg","vader_neu"]].fillna(0)
)

print(f"Sentiment features added. master_df shape: {master_df.shape}")
master_df[["trading_day","ticker","vader_compound","vader_pos","vader_neg","vader_neu"]].head()

## Section 9: Named Entity Recognition (NER) with spaCy

Extract **semantic features** per trading day:
- `ner_entity_count` — total named entities  
- `ner_entity_diversity` — number of unique entity types  
- `ner_org_count` — count of ORG-type entities (companies/organizations)

In [ ]:
def extract_ner_features(text: str) -> dict:
    """Run spaCy NER on text and return aggregate semantic features."""
    doc = nlp(text[:100_000])   # truncate very long strings
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    return {
        "ner_entity_count":     len(entities),
        "ner_entity_diversity": len(set(lbl for _, lbl in entities)),
        "ner_org_count":        sum(1 for _, lbl in entities if lbl == "ORG"),
    }

print("Running NER on aggregated daily text — this may take a minute...")
ner_rows = master_df["daily_text"].apply(extract_ner_features)
ner_df   = pd.DataFrame(ner_rows.tolist())

master_df = pd.concat([master_df.reset_index(drop=True), ner_df], axis=1)

print(f"NER features added. master_df shape: {master_df.shape}")
master_df[["trading_day","ticker","ner_entity_count","ner_entity_diversity","ner_org_count"]].head()

## Section 10: RDF Knowledge Graph Construction

Build an RDF graph using **rdflib** with the following schema:

| Node Type | Description |
|---|---|
| `Company` | AAPL or AMZN |
| `NewsArticle` | Individual news article |
| `Entity` | Named entity extracted via NER |
| `TradingDay` | Calendar date (trading day) |
| `DirectionLabel` | UP or DOWN |

| Edge (Predicate) | Description |
|---|---|
| `MENTIONS` | Company → NewsArticle |
| `HAS_ENTITY` | NewsArticle → Entity |
| `PUBLISHED_ON` | NewsArticle → TradingDay |
| `HAS_DIRECTION` | TradingDay → DirectionLabel |

In [ ]:
# ── Namespace setup ────────────────────────────────────────────────────────────
EX   = Namespace("http://stockkg.example.org/")
PRED = Namespace("http://stockkg.example.org/pred/")

kg = Graph()
kg.bind("ex",   EX)
kg.bind("pred", PRED)

# ── Node types ─────────────────────────────────────────────────────────────────
Company       = EX.Company
NewsArticle   = EX.NewsArticle
Entity        = EX.Entity
TradingDay    = EX.TradingDay
DirectionLabel= EX.DirectionLabel

# ── Predicates ─────────────────────────────────────────────────────────────────
MENTIONS       = PRED.MENTIONS
HAS_ENTITY     = PRED.HAS_ENTITY
PUBLISHED_ON   = PRED.PUBLISHED_ON
HAS_DIRECTION  = PRED.HAS_DIRECTION

# ── Populate graph (sample: first 500 rows for speed) ─────────────────────────
MAX_KG_ROWS = 500
sample_df   = master_df.head(MAX_KG_ROWS).copy()

for idx, row in sample_df.iterrows():
    ticker     = row["ticker"]
    day_str    = str(row["trading_day"].date())
    direction  = "UP" if row["label"] == 1 else "DOWN"

    company_node   = URIRef(EX[f"company/{ticker}"])
    article_node   = URIRef(EX[f"article/{idx}"])
    day_node       = URIRef(EX[f"day/{day_str}/{ticker}"])
    direction_node = URIRef(EX[f"direction/{direction}"])

    # Node types
    kg.add((company_node,   RDF.type, Company))
    kg.add((article_node,   RDF.type, NewsArticle))
    kg.add((day_node,       RDF.type, TradingDay))
    kg.add((direction_node, RDF.type, DirectionLabel))

    # Label literals
    kg.add((company_node,   RDFS.label, Literal(ticker)))
    kg.add((day_node,       RDFS.label, Literal(day_str, datatype=XSD.date)))
    kg.add((direction_node, RDFS.label, Literal(direction)))

    # Edges
    kg.add((company_node, MENTIONS,      article_node))
    kg.add((article_node, PUBLISHED_ON,  day_node))
    kg.add((day_node,     HAS_DIRECTION, direction_node))

    # NER entities → HAS_ENTITY edges
    doc = nlp(str(row["daily_text"])[:5000])
    for ent in doc.ents[:10]:   # cap per article
        safe_ent = ent.text.replace(" ", "_").replace("/", "_")
        entity_node = URIRef(EX[f"entity/{safe_ent}_{ent.label_}"])
        kg.add((entity_node, RDF.type,   Entity))
        kg.add((entity_node, RDFS.label, Literal(f"{ent.text} [{ent.label_}]")))
        kg.add((article_node, HAS_ENTITY, entity_node))

print(f"✅ Knowledge Graph: {len(kg):,} triples")

# ── Serialize to Turtle ────────────────────────────────────────────────────────
turtle_str = kg.serialize(format="turtle")
print("\n── Sample Turtle (first 60 lines) ──────────────────────────────────")
for line in turtle_str.splitlines()[:60]:
    print(line)

## Section 11: SPARQL Queries on the Knowledge Graph

Query the RDF graph for explainability insights:
1. Entity → Direction relationships  
2. Most mentioned ORG entities on UP vs DOWN days  
3. Article → TradingDay links

In [ ]:
PREFIX_STR = """
PREFIX ex:   <http://stockkg.example.org/>
PREFIX pred: <http://stockkg.example.org/pred/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
"""

# ── Query 1: Entity–Direction pairs ───────────────────────────────────────────
q1 = PREFIX_STR + """
SELECT ?entityLabel ?direction
WHERE {
    ?article rdf:type ex:NewsArticle ;
             pred:HAS_ENTITY  ?entity ;
             pred:PUBLISHED_ON ?day .
    ?day     pred:HAS_DIRECTION ?dir .
    ?entity  rdfs:label ?entityLabel .
    ?dir     rdfs:label ?direction .
}
LIMIT 20
"""
print("── Query 1: Entity → Direction ─────────────────────────────────────")
for row in kg.query(q1):
    print(f"  {str(row.entityLabel):<40} → {str(row.direction)}")

# ── Query 2: Most mentioned entities on UP days ───────────────────────────────
q2 = PREFIX_STR + """
SELECT ?entityLabel (COUNT(?article) AS ?cnt)
WHERE {
    ?article rdf:type ex:NewsArticle ;
             pred:HAS_ENTITY  ?entity ;
             pred:PUBLISHED_ON ?day .
    ?day     pred:HAS_DIRECTION ?dir .
    ?dir     rdfs:label "UP" .
    ?entity  rdfs:label ?entityLabel .
}
GROUP BY ?entityLabel
ORDER BY DESC(?cnt)
LIMIT 10
"""
print("\n── Query 2: Top entities on UP days ────────────────────────────────")
for row in kg.query(q2):
    print(f"  {str(row.entityLabel):<40} count={int(row.cnt)}")

# ── Query 3: Articles published on a specific day ─────────────────────────────
sample_day = str(master_df["trading_day"].iloc[0].date())
q3 = PREFIX_STR + f"""
SELECT ?article ?day
WHERE {{
    ?article rdf:type ex:NewsArticle ;
             pred:PUBLISHED_ON ?day .
    ?day rdfs:label "{sample_day}"^^<http://www.w3.org/2001/XMLSchema#date> .
}}
LIMIT 10
"""
print(f"\n── Query 3: Articles on {sample_day} ─────────────────────────────")
results = list(kg.query(q3))
print(f"  Found {len(results)} article(s)")

## Section 12: Feature Matrix Assembly

Horizontally concatenate:
1. **TF-IDF** (chi-square selected, sparse)
2. **VADER** sentiment (4 dense features)
3. **NER** semantic features (3 dense features)

In [ ]:
SENTIMENT_COLS = ["vader_compound", "vader_pos", "vader_neg", "vader_neu"]
SEMANTIC_COLS  = ["ner_entity_count", "ner_entity_diversity", "ner_org_count"]

X_sent = csr_matrix(master_df[SENTIMENT_COLS].fillna(0).values)
X_sem  = csr_matrix(master_df[SEMANTIC_COLS].fillna(0).values)
y      = master_df["label"].values

# Full feature matrix: TF-IDF + Sentiment + Semantic
X_full = hstack([X_tfidf_sel, X_sent, X_sem])

# Partial matrices for ablation
X_tfidf_only      = X_tfidf_sel
X_tfidf_sent      = hstack([X_tfidf_sel, X_sent])
X_tfidf_sent_sem  = X_full

print(f"X_tfidf_only     : {X_tfidf_only.shape}")
print(f"X_tfidf_sent     : {X_tfidf_sent.shape}")
print(f"X_tfidf_sent_sem : {X_tfidf_sent_sem.shape}")
print(f"Labels (y)       : {y.shape}  |  class balance: {np.bincount(y)}")

# NaN check
import numpy as np
assert not np.isnan(X_full.data).any(), "NaN values in feature matrix!"
print("✅ No NaN values in feature matrix.")

## Section 13: Chronological Train / Test Split

- **Train:** first ~80% of trading days (chronological order)
- **Test:** remaining ~20% (held-out, no shuffling)

In [ ]:
n = len(y)
split_idx = int(n * 0.80)   # 80/20 chronological split

X_train = X_full[:split_idx]
X_test  = X_full[split_idx:]
y_train = y[:split_idx]
y_test  = y[split_idx:]

print(f"Total samples : {n}")
print(f"Train size    : {X_train.shape[0]}  |  class dist: {np.bincount(y_train)}")
print(f"Test  size    : {X_test.shape[0]}   |  class dist: {np.bincount(y_test)}")

# Store split index for ablation reuse
train_dates = master_df["trading_day"].iloc[:split_idx]
test_dates  = master_df["trading_day"].iloc[split_idx:]
print(f"\nTrain date range: {train_dates.min().date()} → {train_dates.max().date()}")
print(f"Test  date range: {test_dates.min().date()}  → {test_dates.max().date()}")

## Section 14: Classical Baseline Models — Logistic Regression (L1 & L2) and Naïve Bayes

Train three classical baselines:
- **Logistic Regression L1** (Lasso — sparsity, feature selection)
- **Logistic Regression L2** (Ridge — shrinkage)
- **Multinomial Naïve Bayes** (fast probabilistic baseline from Alostad & Davulcu 2017)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import issparse

# MinMax scale for NB (requires non-negative input)
def to_nonneg(X):
    """Shift sparse matrix so all values >= 0 for MultinomialNB."""
    if issparse(X):
        X = X.toarray()
    X = X - X.min(axis=0)
    return X

# ── Logistic Regression L1 ────────────────────────────────────────────────────
lr_l1 = LogisticRegression(
    penalty="l1", solver="saga", C=1.0,
    class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
)
lr_l1.fit(X_train, y_train)
y_pred_l1 = lr_l1.predict(X_test)

# ── Logistic Regression L2 ────────────────────────────────────────────────────
lr_l2 = LogisticRegression(
    penalty="l2", solver="lbfgs", C=1.0,
    class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
)
lr_l2.fit(X_train, y_train)
y_pred_l2 = lr_l2.predict(X_test)

# ── Multinomial Naïve Bayes ───────────────────────────────────────────────────
X_train_nn = to_nonneg(X_train)
X_test_nn  = to_nonneg(X_test)
nb = MultinomialNB(alpha=1.0)
nb.fit(X_train_nn, y_train)
y_pred_nb = nb.predict(X_test_nn)

print("✅ Classical models trained.")
for name, pred in [("LR-L1", y_pred_l1), ("LR-L2", y_pred_l2), ("NaïveBayes", y_pred_nb)]:
    f1 = f1_score(y_test, pred, average="macro", zero_division=0)
    acc = accuracy_score(y_test, pred)
    print(f"  {name:<12}  Acc={acc:.3f}  F1={f1:.3f}")

## Section 15: Deep Learning — BERT / FinBERT Embeddings

Use `sentence-transformers` to generate **BERT sentence embeddings** for each day's aggregated text. These dense vectors serve as input to a fine-tuned classifier.

> **Note:** Requires `pip install sentence-transformers transformers torch`. For FinBERT, use `ProsusAI/finbert`.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    BERT_AVAILABLE = True
except ImportError:
    BERT_AVAILABLE = False
    print("⚠️  sentence-transformers not installed. Run: pip install sentence-transformers")
    print("   Generating random BERT-shaped embeddings as placeholder.")

BERT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # lightweight; swap for ProsusAI/finbert

if BERT_AVAILABLE:
    print(f"Loading BERT model: {BERT_MODEL_NAME} …")
    sbert = SentenceTransformer(BERT_MODEL_NAME)
    texts = master_df["daily_text"].tolist()
    # Truncate each text to 512 tokens worth of characters
    texts_trunc = [t[:2000] for t in texts]
    X_bert = sbert.encode(texts_trunc, batch_size=32, show_progress_bar=True,
                          convert_to_numpy=True)
    print(f"✅ BERT embeddings shape: {X_bert.shape}")
else:
    # Placeholder: 384-dim (same as MiniLM) random unit vectors
    rng = np.random.default_rng(42)
    X_bert = rng.standard_normal((len(master_df), 384)).astype(np.float32)
    X_bert = X_bert / np.linalg.norm(X_bert, axis=1, keepdims=True)
    print(f"Placeholder BERT embeddings shape: {X_bert.shape}")

# Train/test split for BERT features
X_bert_train = X_bert[:split_idx]
X_bert_test  = X_bert[split_idx:]

# Fine-tune a Logistic Regression head on BERT embeddings
lr_bert = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
lr_bert.fit(X_bert_train, y_train)
y_pred_bert = lr_bert.predict(X_bert_test)

f1_bert  = f1_score(y_test, y_pred_bert, average="macro", zero_division=0)
acc_bert = accuracy_score(y_test, y_pred_bert)
print(f"\nBERT + LogReg head:  Acc={acc_bert:.3f}  F1={f1_bert:.3f}")

## Section 16: Deep Learning — LSTM Sequential Model

A **BiLSTM** processes BERT embeddings as a daily time series, capturing temporal dependencies in news flow.

In [ ]:
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠️  PyTorch not installed. Run: pip install torch")

WINDOW = 5   # look-back window of 5 trading days

def make_sequences(X_arr, y_arr, window=WINDOW):
    """Slide a window over the time series to create (seq, label) pairs."""
    Xs, ys = [], []
    for i in range(window, len(y_arr)):
        Xs.append(X_arr[i - window : i])
        ys.append(y_arr[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.int64)

if TORCH_AVAILABLE:
    class BiLSTMClassifier(nn.Module):
        def __init__(self, input_size, hidden_size=64, num_layers=1, dropout=0.3):
            super().__init__()
            self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                                batch_first=True, bidirectional=True, dropout=dropout if num_layers > 1 else 0)
            self.fc   = nn.Linear(hidden_size * 2, 2)
            self.drop = nn.Dropout(dropout)

        def forward(self, x):
            _, (h, _) = self.lstm(x)
            h = torch.cat([h[-2], h[-1]], dim=1)   # both directions
            return self.fc(self.drop(h))

    Xs_all, ys_all = make_sequences(X_bert, y)
    n_seq   = len(ys_all)
    tr_end  = int(n_seq * 0.80)
    Xs_tr, ys_tr = Xs_all[:tr_end], ys_all[:tr_end]
    Xs_te, ys_te = Xs_all[tr_end:], ys_all[tr_end:]

    loader = DataLoader(TensorDataset(torch.from_numpy(Xs_tr), torch.from_numpy(ys_tr)),
                        batch_size=32, shuffle=False)

    model   = BiLSTMClassifier(input_size=X_bert.shape[1])
    optim   = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    EPOCHS = 10
    model.train()
    for epoch in range(1, EPOCHS + 1):
        total_loss = 0
        for xb, yb in loader:
            optim.zero_grad()
            out  = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            optim.step()
            total_loss += loss.item()
        if epoch % 2 == 0:
            print(f"  Epoch {epoch}/{EPOCHS}  loss={total_loss/len(loader):.4f}")

    model.eval()
    with torch.no_grad():
        logits = model(torch.from_numpy(Xs_te))
        y_pred_lstm = logits.argmax(dim=1).numpy()

    f1_lstm  = f1_score(ys_te, y_pred_lstm, average="macro", zero_division=0)
    acc_lstm = accuracy_score(ys_te, y_pred_lstm)
    print(f"\n✅ BiLSTM:  Acc={acc_lstm:.3f}  F1={f1_lstm:.3f}")

else:
    # Fallback: report placeholder metrics
    y_pred_lstm = np.zeros(len(y_test), dtype=int)
    f1_lstm  = 0.0
    acc_lstm = 0.0
    print("BiLSTM skipped (PyTorch not available). Install torch to enable.")

## Section 17: TimeSeriesSplit Cross-Validation

5-fold chronological cross-validation on the **training set** for all models.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

models_cv = {
    "LR-L1":      LogisticRegression(penalty="l1", solver="saga",  C=1.0, class_weight="balanced", max_iter=500, random_state=RANDOM_STATE),
    "LR-L2":      LogisticRegression(penalty="l2", solver="lbfgs", C=1.0, class_weight="balanced", max_iter=500, random_state=RANDOM_STATE),
    "NaïveBayes": None,   # handled separately
    "BERT+LR":    LogisticRegression(C=1.0, class_weight="balanced", max_iter=500, random_state=RANDOM_STATE),
}

cv_results = {name: {"acc": [], "f1": []} for name in models_cv}

for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_train), 1):
    Xtr, Xva = X_train[tr_idx], X_train[va_idx]
    ytr, yva = y_train[tr_idx], y_train[va_idx]
    Btr, Bva = X_bert_train[tr_idx], X_bert_train[va_idx]

    for name, clf in models_cv.items():
        if name == "NaïveBayes":
            clf_nb = MultinomialNB(alpha=1.0)
            clf_nb.fit(to_nonneg(Xtr), ytr)
            pred = clf_nb.predict(to_nonneg(Xva))
        elif name == "BERT+LR":
            clf.fit(Btr, ytr)
            pred = clf.predict(Bva)
        else:
            clf.fit(Xtr, ytr)
            pred = clf.predict(Xva)

        cv_results[name]["acc"].append(accuracy_score(yva, pred))
        cv_results[name]["f1"].append(f1_score(yva, pred, average="macro", zero_division=0))

print(f"{'Model':<14}  {'Acc mean±std':>18}  {'F1 mean±std':>18}")
print("-" * 56)
for name, res in cv_results.items():
    accs = np.array(res["acc"])
    f1s  = np.array(res["f1"])
    print(f"{name:<14}  {accs.mean():.3f} ± {accs.std():.3f}        {f1s.mean():.3f} ± {f1s.std():.3f}")

## Section 18: Model Evaluation & Metrics

Full held-out test set evaluation for all models with confusion matrices.

In [ ]:
test_preds = {
    "LR-L1":      y_pred_l1,
    "LR-L2":      y_pred_l2,
    "NaïveBayes": y_pred_nb,
    "BERT+LR":    y_pred_bert,
}

summary_rows = []
for name, pred in test_preds.items():
    row = {
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_test, pred), 4),
        "Precision": round(precision_score(y_test, pred, average="macro", zero_division=0), 4),
        "Recall":    round(recall_score(y_test, pred, average="macro", zero_division=0), 4),
        "F1 (macro)":round(f1_score(y_test, pred, average="macro", zero_division=0), 4),
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("Model")
print(summary_df.to_string())

# ── Confusion matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(test_preds), figsize=(5 * len(test_preds), 4))
if len(test_preds) == 1:
    axes = [axes]

for ax, (name, pred) in zip(axes, test_preds.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["DOWN", "UP"], yticklabels=["DOWN", "UP"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices — Test Set", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Section 19: Ablation Study

Compare **5 feature configurations** across models to quantify the contribution of each feature group.

| Config | Features |
|---|---|
| A | TF-IDF only |
| B | TF-IDF + Sentiment |
| C | TF-IDF + Sentiment + Semantic |
| D | BERT embeddings only |
| E | BERT + Sentiment + Semantic (full) |

In [ ]:
X_bert_dense    = csr_matrix(X_bert)
X_bert_full     = hstack([X_bert_dense, X_sent, X_sem])

ablation_configs = {
    "A: TF-IDF":                   X_tfidf_only,
    "B: TF-IDF+Sent":              X_tfidf_sent,
    "C: TF-IDF+Sent+Sem":          X_tfidf_sent_sem,
    "D: BERT":                     X_bert_dense,
    "E: BERT+Sent+Sem":            X_bert_full,
}

ablation_results = []
for config_name, X_cfg in ablation_configs.items():
    Xtr_a = X_cfg[:split_idx]
    Xte_a = X_cfg[split_idx:]

    # LR-L2 as representative classifier
    clf_a = LogisticRegression(penalty="l2", solver="lbfgs", C=1.0,
                               class_weight="balanced", max_iter=500, random_state=RANDOM_STATE)
    clf_a.fit(Xtr_a, y_train)
    pred_a = clf_a.predict(Xte_a)

    ablation_results.append({
        "Config":      config_name,
        "Accuracy":    round(accuracy_score(y_test, pred_a), 4),
        "F1 (macro)":  round(f1_score(y_test, pred_a, average="macro", zero_division=0), 4),
    })

abl_df = pd.DataFrame(ablation_results)
print(abl_df.to_string(index=False))

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(abl_df))
w = 0.35
bars1 = ax.bar(x - w/2, abl_df["Accuracy"],  w, label="Accuracy",   color="#4C72B0")
bars2 = ax.bar(x + w/2, abl_df["F1 (macro)"],w, label="F1 (macro)", color="#DD8452")
ax.set_xticks(x)
ax.set_xticklabels(abl_df["Config"], rotation=15, ha="right", fontsize=9)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Ablation Study: Feature Configuration vs. Performance (LR-L2)")
ax.legend()
plt.tight_layout()
plt.show()

## Section 20: Interpretability — Coefficient Analysis

Map top LR coefficients back to feature names to identify the most predictive keywords and semantic signals.

In [ ]:
feature_names = list(selected_names) + SENTIMENT_COLS + SEMANTIC_COLS

def plot_top_coefs(clf, feature_names, title, top_n=20):
    coefs = clf.coef_[0]
    idx_top  = np.argsort(coefs)[-top_n:][::-1]
    idx_bot  = np.argsort(coefs)[:top_n]
    combined = np.concatenate([idx_top, idx_bot])
    labels   = [feature_names[i] if i < len(feature_names) else f"feat_{i}" for i in combined]
    values   = coefs[combined]
    colors   = ["#2ecc71" if v > 0 else "#e74c3c" for v in values]

    fig, ax = plt.subplots(figsize=(9, max(6, top_n * 0.4)))
    ax.barh(range(len(labels)), values, color=colors)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Coefficient value")
    ax.set_title(title)
    green_p = mpatches.Patch(color="#2ecc71", label="→ UP")
    red_p   = mpatches.Patch(color="#e74c3c", label="→ DOWN")
    ax.legend(handles=[green_p, red_p], loc="lower right")
    plt.tight_layout()
    plt.show()

# Retrain on full feature matrix so coefs cover all feature names
lr_l1_full = LogisticRegression(penalty="l1", solver="saga",  C=1.0, class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
lr_l2_full = LogisticRegression(penalty="l2", solver="lbfgs", C=1.0, class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
lr_l1_full.fit(X_train, y_train)
lr_l2_full.fit(X_train, y_train)

plot_top_coefs(lr_l1_full, feature_names, "LR-L1: Top 20 Positive & Negative Predictive Features")
plot_top_coefs(lr_l2_full, feature_names, "LR-L2: Top 20 Positive & Negative Predictive Features")

## Section 21: Summary Visualizations

Final dashboard of key visualizations for the project report.

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.suptitle("Directional Stock Prediction — Summary Dashboard", fontsize=15, y=1.01)

# ── (1) Daily VADER compound sentiment over time ───────────────────────────────
ax1 = fig.add_subplot(3, 3, 1)
for ticker in TICKERS:
    sub = master_df[master_df["ticker"] == ticker].sort_values("trading_day")
    ax1.plot(sub["trading_day"], sub["vader_compound"], label=ticker, alpha=0.7, linewidth=0.8)
ax1.axhline(0, color="gray", linewidth=0.6, linestyle="--")
ax1.set_title("Daily VADER Compound Sentiment")
ax1.set_xlabel("Date")
ax1.set_ylabel("Compound score")
ax1.legend(fontsize=8)
ax1.tick_params(axis="x", rotation=30)

# ── (2) Class label distribution ──────────────────────────────────────────────
ax2 = fig.add_subplot(3, 3, 2)
counts = pd.Series(y_all).value_counts().sort_index()
ax2.bar(["DOWN (0)", "UP (1)"], counts.values, color=["#e74c3c", "#2ecc71"])
ax2.set_title("Class Label Distribution")
ax2.set_ylabel("Count")
for i, v in enumerate(counts.values):
    ax2.text(i, v + 1, str(v), ha="center", fontsize=10)

# ── (3) NER entity count: UP vs DOWN days ─────────────────────────────────────
ax3 = fig.add_subplot(3, 3, 3)
sns.boxplot(data=master_df, x="label", y="ner_entity_count", ax=ax3,
            palette={0: "#e74c3c", 1: "#2ecc71"})
ax3.set_xticklabels(["DOWN", "UP"])
ax3.set_title("NER Entity Count: UP vs DOWN Days")
ax3.set_xlabel("Direction")
ax3.set_ylabel("Entity count")

# ── (4) NER ORG count: UP vs DOWN days ────────────────────────────────────────
ax4 = fig.add_subplot(3, 3, 4)
sns.boxplot(data=master_df, x="label", y="ner_org_count", ax=ax4,
            palette={0: "#e74c3c", 1: "#2ecc71"})
ax4.set_xticklabels(["DOWN", "UP"])
ax4.set_title("ORG Entity Count: UP vs DOWN Days")
ax4.set_xlabel("Direction")
ax4.set_ylabel("ORG count")

# ── (5) Ablation bar chart ─────────────────────────────────────────────────────
ax5 = fig.add_subplot(3, 3, 5)
x5 = np.arange(len(abl_df))
ax5.bar(x5 - 0.2, abl_df["Accuracy"],  0.35, label="Accuracy",   color="#4C72B0")
ax5.bar(x5 + 0.2, abl_df["F1 (macro)"],0.35, label="F1 (macro)", color="#DD8452")
ax5.set_xticks(x5)
ax5.set_xticklabels([c.split(":")[0] for c in abl_df["Config"]], fontsize=8)
ax5.set_ylim(0, 1)
ax5.set_title("Ablation Study (LR-L2)")
ax5.legend(fontsize=8)

# ── (6) Model comparison bar chart ────────────────────────────────────────────
ax6 = fig.add_subplot(3, 3, 6)
model_names = list(summary_df.index)
f1_vals     = summary_df["F1 (macro)"].values
colors6     = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"][:len(model_names)]
ax6.bar(model_names, f1_vals, color=colors6)
ax6.set_ylim(0, 1)
ax6.set_title("F1 (macro) — All Models")
ax6.set_ylabel("F1 Score")
ax6.tick_params(axis="x", rotation=15)
for i, v in enumerate(f1_vals):
    ax6.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)

# ── (7) Knowledge Graph node–edge diagram (networkx) ──────────────────────────
ax7 = fig.add_subplot(3, 3, 7)
G = nx.DiGraph()
nodes = {
    "AAPL":           "Company",
    "AMZN":           "Company",
    "Article_1":      "NewsArticle",
    "Apple [ORG]":    "Entity",
    "2018-03-05":     "TradingDay",
    "UP":             "DirectionLabel",
}
color_map = {"Company": "#3498db", "NewsArticle": "#f39c12",
             "Entity": "#9b59b6", "TradingDay": "#1abc9c", "DirectionLabel": "#2ecc71"}
for n, t in nodes.items():
    G.add_node(n, ntype=t)
edges_kg = [
    ("AAPL",       "Article_1",   "MENTIONS"),
    ("Article_1",  "Apple [ORG]", "HAS_ENTITY"),
    ("Article_1",  "2018-03-05",  "PUBLISHED_ON"),
    ("2018-03-05", "UP",          "HAS_DIRECTION"),
]
for u, v, lbl in edges_kg:
    G.add_edge(u, v, label=lbl)

pos = nx.spring_layout(G, seed=42, k=1.5)
node_colors = [color_map[nodes[n]] for n in G.nodes()]
nx.draw_networkx(G, pos, ax=ax7, node_color=node_colors, node_size=1200,
                 font_size=7, arrows=True, arrowsize=15, edge_color="#7f8c8d",
                 connectionstyle="arc3,rad=0.1")
edge_labels = nx.get_edge_attributes(G, "label")
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax7,
                              font_size=6, label_pos=0.5)
ax7.set_title("Knowledge Graph (Schema Sample)")
ax7.axis("off")
legend_patches = [mpatches.Patch(color=c, label=t) for t, c in color_map.items()]
ax7.legend(handles=legend_patches, loc="lower left", fontsize=6)

# ── (8) TimeSeriesSplit CV F1 per fold ────────────────────────────────────────
ax8 = fig.add_subplot(3, 3, 8)
for name, res in cv_results.items():
    ax8.plot(range(1, 6), res["f1"], marker="o", label=name)
ax8.set_title("TimeSeriesSplit CV — F1 per Fold")
ax8.set_xlabel("Fold")
ax8.set_ylabel("F1 (macro)")
ax8.set_ylim(0, 1)
ax8.legend(fontsize=7)

# ── (9) Stock closing price over time ─────────────────────────────────────────
ax9 = fig.add_subplot(3, 3, 9)
for ticker in TICKERS:
    df_s = stock_frames[ticker].reset_index()
    ax9.plot(df_s["date"], df_s["Close"], label=ticker)
ax9.set_title("DJI Ticker Closing Prices")
ax9.set_xlabel("Date")
ax9.set_ylabel("Close ($)")
ax9.legend()
ax9.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("summary_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Dashboard saved to summary_dashboard.png")